In [ ]:
%pip install -q -r ../requirements.txt

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import HfApi, login
from IPython.display import display

warnings.filterwarnings('ignore')

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.local.classification.vlm.models import (load_openai_clip, load_biomedclip, load_unimedclip, make_openai_clip_loader, make_biomedclip_loader, make_unimedclip_loader)
from src.local.classification.vlm.helpers import (load_busi_splits, save_results_to_csv, encode_images_batch)
from src.local.classification.vlm.zero_shot import ZeroShotEvaluator
from src.local.prompting.prompt_registry import PROMPT_REGISTRY
from src.local.classification.vlm.few_shot_lp import (LinearProbeConfig, make_kshot_indices, run_linear_probe_experiments)
from src.local.classification.vlm.few_shot_lora import get_args, run_kshot_experiments

c:\Users\mason\Desktop\busi-vlm-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(project_root / '.env', override=True)

huggingface_token = os.getenv('huggingface_token')
api = HfApi(token=huggingface_token) if huggingface_token else None

if huggingface_token:
    login(token=huggingface_token, add_to_git_credential=False)
    print('HuggingFace authenticated as -', api.whoami()['name'])
else:
    print('No huggingface_token found.')

HuggingFace authenticated as - masonezra34


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device -', device)

if device == 'cuda':
    print(torch.cuda.get_device_name(0))

device - cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
# Setup BUSI classes and prompts.
busi_classes = ['benign', 'malignant', 'normal']

class_mapping = {
    'benign': 'benign tumor',
    'malignant': 'malignant tumor',
    'normal': 'normal scan'
}

class_names_for_prompts = [class_mapping[c] for c in busi_classes]

print(f'BUSI classes - {busi_classes}')
print(f'Prompt registry classes - {class_names_for_prompts}')
print(f'Prompts per class:')
for class_name in class_names_for_prompts:
    print(f'  {class_name}: {len(PROMPT_REGISTRY[class_name])} prompts')

# Load BUSI train/val/test splits.
train_df, val_df, test_df = load_busi_splits(project_root, busi_classes)

print(f'\ndataset splits:')
print(f'* train - {len(train_df)}')
print(f'* validation - {len(val_df)}')
print(f'* test - {len(test_df)}')
print(f'\ntest distribution:')
print(test_df['label'].value_counts())

BUSI classes - ['benign', 'malignant', 'normal']
Prompt registry classes - ['benign tumor', 'malignant tumor', 'normal scan']
Prompts per class:
  benign tumor: 8 prompts
  malignant tumor: 8 prompts
  normal scan: 8 prompts

dataset splits:
* train - 437
* validation - 93
* test - 95

test distribution:
label
benign       51
malignant    29
normal       15
Name: count, dtype: int64


In [5]:
# Load OpenAI CLIP ViT-B/16.
openai_model, openai_preprocess, openai_tokenizer = load_openai_clip(device=device)

# Create evaluator.
openai_evaluator = ZeroShotEvaluator(
    openai_model,
    openai_preprocess,
    openai_tokenizer,
    PROMPT_REGISTRY,
    class_names_for_prompts,
    device=device
)

# Build text embeddings.
openai_evaluator.build_text_embeddings()

# Evaluate on test set.
openai_metrics, _, _ = openai_evaluator.evaluate(
    test_df,
    batch_size=32,
    temperature=100.0,
    description='encoding test images (OpenAI CLIP)'
)

# Print results.
openai_evaluator.print_results(openai_metrics, 'OpenAI CLIP ViT-B/16')

# Save results.
results_path = project_root / 'results' / 'zero_shot_results.csv'
results_path.parent.mkdir(exist_ok=True)
save_results_to_csv(openai_metrics, results_path, 'openai_clip_vit_b16', append=False)
print(f'\nresults saved to - {results_path}')

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (OpenAI CLIP): 100%|██████████| 3/3 [00:01<00:00,  1.87it/s]


Zero-Shot Results: OpenAI CLIP ViT-B/16
accuracy - 0.5684
balanced Accuracy - 0.3728
macro F1 - 0.3148
AUC - 0.7215

per-class F1 -
  benign: 0.7092
  malignant: 0.2353
  normal: 0.0000

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


In [6]:
# Load BiomedCLIP ViT-B/16.
biomedclip_model, _, biomedclip_preprocess, biomedclip_tokenizer = load_biomedclip(device=device)

# Create evaluator.
biomedclip_evaluator = ZeroShotEvaluator(
    biomedclip_model,
    biomedclip_preprocess,
    biomedclip_tokenizer,
    PROMPT_REGISTRY,
    class_names_for_prompts,
    device=device,
)

# Build text embeddings.
biomedclip_evaluator.build_text_embeddings()

# Evaluate on test set.
biomedclip_metrics, _, _ = biomedclip_evaluator.evaluate(
    test_df,
    batch_size=32,
    temperature=100.0,
    description='encoding test images (BiomedCLIP)',
)

# Print results.
biomedclip_evaluator.print_results(biomedclip_metrics, 'BiomedCLIP ViT-B/16')

# Save results.
save_results_to_csv(
    biomedclip_metrics,
    results_path,
    'biomedclip_vit_b16',
    append=True,
)

print(f'\nresults saved to - {results_path}')

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (BiomedCLIP): 100%|██████████| 3/3 [00:01<00:00,  2.34it/s]


Zero-Shot Results: BiomedCLIP ViT-B/16
accuracy - 0.4211
balanced Accuracy - 0.4680
macro F1 - 0.3706
AUC - 0.7643

per-class F1 -
  benign: 0.2414
  malignant: 0.5370
  normal: 0.3333

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


In [7]:
# Load UniMed-CLIP ViT-B/16.
unimedclip_model, unimedclip_preprocess, unimedclip_tokenizer = load_unimedclip(
    device=device,
    project_root=project_root,
)

# Create evaluator.
unimedclip_evaluator = ZeroShotEvaluator(
    unimedclip_model,
    unimedclip_preprocess,
    unimedclip_tokenizer,
    PROMPT_REGISTRY,
    class_names_for_prompts,
    device=device
)

# Build text embeddings.
unimedclip_evaluator.build_text_embeddings()

# Evaluate on test set.
unimedclip_metrics, _, _ = unimedclip_evaluator.evaluate(
    test_df,
    batch_size=32,
    temperature=100.0,
    description='encoding test images (UniMed-CLIP)'
)

# Print results.
unimedclip_evaluator.print_results(unimedclip_metrics, 'UniMed-CLIP ViT-B/16')

# Save results.
save_results_to_csv(unimedclip_metrics, results_path, 'unimedclip', append=True)
print(f'\nresults saved to - {results_path}')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24446.81it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be

benign tumor - torch.Size([8, 512])
malignant tumor - torch.Size([8, 512])
normal scan - torch.Size([8, 512])


encoding test images (UniMed-CLIP): 100%|██████████| 3/3 [00:01<00:00,  2.15it/s]


Zero-Shot Results: UniMed-CLIP ViT-B/16
accuracy - 0.3263
balanced Accuracy - 0.4231
macro F1 - 0.3171
AUC - 0.6230

per-class F1 -
  benign: 0.2069
  malignant: 0.4444
  normal: 0.3000

results saved to - c:\Users\mason\Desktop\busi-vlm-project\results\zero_shot_results.csv


In [8]:
# Display zero-shot comparison.
zero_shot_results = pd.read_csv(results_path)
display(zero_shot_results)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,auc,benign_precision,benign_recall,benign_f1,malignant_precision,malignant_recall,malignant_f1,normal_precision,normal_recall,normal_f1
0,openai_clip_vit_b16,0.568421,0.372774,0.314838,0.452566,0.721462,0.555556,0.980392,0.709220,0.800000,0.137931,0.235294,0.000000,0.000000,0.000000
1,biomedclip_vit_b16,0.421053,0.467974,0.370583,0.346152,0.764309,1.000000,0.137255,0.241379,0.367089,1.000000,0.537037,0.444444,0.266667,0.333333
2,unimedclip,0.326316,0.423124,0.317114,0.294112,0.623004,0.857143,0.117647,0.206897,0.372093,0.551724,0.444444,0.200000,0.600000,0.300000


In [9]:
feature_batch_size = 32

model_specs = [
    {
        'model_name': 'openai_clip_vit_b16',
        'model': openai_model,
        'preprocess': openai_preprocess,
    },
    {
        'model_name': 'biomedclip_vit_b16',
        'model': biomedclip_model,
        'preprocess': biomedclip_preprocess,
    },
    {
        'model_name': 'unimedclip',
        'model': unimedclip_model,
        'preprocess': unimedclip_preprocess,
    },
]

feature_sets = {}

for spec in model_specs:
    model_name = spec['model_name']
    model = spec['model']
    preprocess = spec['preprocess']

    print(f'encoding frozen features - {model_name}')

    model.eval()

    feature_sets[model_name] = {
        'train': encode_images_batch(
            model,
            preprocess,
            train_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} train',
        ),
        'val': encode_images_batch(
            model,
            preprocess,
            val_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} val',
        ),
        'test': encode_images_batch(
            model,
            preprocess,
            test_df,
            device=device,
            batch_size=feature_batch_size,
            description=f'{model_name} test',
        ),
    }

    print(
        f"{model_name} features - "
        f"train {feature_sets[model_name]['train'].shape}, "
        f"val {feature_sets[model_name]['val'].shape}, "
        f"test {feature_sets[model_name]['test'].shape}"
    )

train_labels = train_df['label_index'].values
val_labels = val_df['label_index'].values
test_labels = test_df['label_index'].values

encoding frozen features - openai_clip_vit_b16


openai_clip_vit_b16 test: 100%|██████████| 3/3 [00:01<00:00,  2.22it/s]


openai_clip_vit_b16 features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])
encoding frozen features - biomedclip_vit_b16


biomedclip_vit_b16 test: 100%|██████████| 3/3 [00:01<00:00,  2.33it/s]


biomedclip_vit_b16 features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])
encoding frozen features - unimedclip


unimedclip test: 100%|██████████| 3/3 [00:01<00:00,  2.20it/s]

unimedclip features - train torch.Size([437, 512]), val torch.Size([93, 512]), test torch.Size([95, 512])


In [ ]:
# Few-shot setup.
shots_per_class = [1, 2, 4, 8, 16, 32]
seeds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

shared_support_indices = make_kshot_indices(
    train_df,
    label_col='label_index',
    shots_per_class=shots_per_class,
    seeds=seeds,
)

In [11]:
# Few-shot linear probe.
probe_config = LinearProbeConfig(max_iter=5000, class_weight='balanced', c_values=(0.001, 0.01, 0.1, 1.0, 10.0), selection_metric='macro_f1')

all_results = []
all_aggregates = []

for spec in model_specs:
    model_name = spec['model_name']
    features = feature_sets[model_name]

    print(f'few-shot linear probe - {model_name}')

    results_df, aggregate_df = run_linear_probe_experiments(
        model_name=model_name,
        train_features=features["train"],
        train_labels=train_labels,
        val_features=features["val"],
        val_labels=val_labels,
        test_features=features["test"],
        test_labels=test_labels,
        class_names=busi_classes,
        ratios=shots_per_class,
        seeds=seeds,
        device=device,
        config=probe_config,
        verbose=False,
        kshot_indices=shared_support_indices
    )

    all_results.append(results_df)
    all_aggregates.append(aggregate_df)

fewshot_results = pd.concat(all_results, ignore_index=True)
fewshot_aggregate = pd.concat(all_aggregates, ignore_index=True)

summary_columns = [
    'model',
    'shots_per_class',
    'n_train_samples_mean',
    'accuracy_mean',
    'accuracy_std',
    'balanced_accuracy_mean',
    'balanced_accuracy_std',
    'macro_f1_mean',
    'macro_f1_std',
    'auc_mean',
    'auc_std'
]

display(fewshot_aggregate[summary_columns])

fewshot_dir = project_root / 'results' / 'fewshot_linear_probe'
fewshot_dir.mkdir(parents=True, exist_ok=True)

fewshot_results_path = fewshot_dir / 'fewshot_linear_probe_all_runs.csv'
fewshot_aggregate_path = fewshot_dir / 'fewshot_linear_probe_aggregate.csv'

fewshot_results.to_csv(fewshot_results_path, index=False)
fewshot_aggregate.to_csv(fewshot_aggregate_path, index=False)

print(f'few-shot runs saved to - {fewshot_results_path}')
print(f'few-shot aggregate saved to - {fewshot_aggregate_path}')

few-shot linear probe - openai_clip_vit_b16
model=openai_clip_vit_b16 shots=1 seed=1 | n=3 | c=10.0 | val_f1=0.3037 | test_acc=0.2632 | test_f1=0.2490 | test_auc=0.6649
few-shot linear probe - biomedclip_vit_b16
model=biomedclip_vit_b16 shots=1 seed=1 | n=3 | c=10.0 | val_f1=0.3998 | test_acc=0.5474 | test_f1=0.4426 | test_auc=0.5926
few-shot linear probe - unimedclip
model=unimedclip shots=1 seed=1 | n=3 | c=10.0 | val_f1=0.4165 | test_acc=0.3053 | test_f1=0.3017 | test_auc=0.4977


,model,shots_per_class,n_train_samples_mean,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,auc_mean,auc_std
0,openai_clip_vit_b16,1,3.0,0.263158,NaN,0.366509,NaN,0.249006,NaN,0.664853,NaN
1,biomedclip_vit_b16,1,3.0,0.547368,NaN,0.458778,NaN,0.442647,NaN,0.592647,NaN
2,unimedclip,1,3.0,0.305263,NaN,0.343160,NaN,0.301686,NaN,0.497656,NaN


few-shot runs saved to - c:\Users\mason\Desktop\busi-vlm-project\results\fewshot_linear_probe\fewshot_linear_probe_all_runs.csv
few-shot aggregate saved to - c:\Users\mason\Desktop\busi-vlm-project\results\fewshot_linear_probe\fewshot_linear_probe_aggregate.csv


In [12]:
# Use BUSI class names here (not prompt-mapped names).
lora_class_names = busi_classes

lora_base_save_dir = project_root/'results'/'few_shot_lora'
lora_base_save_dir.mkdir(parents=True, exist_ok=True)

In [13]:
# Run LoRA training for OpenAI CLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False

args.model_name = 'openai_clip_vit_b16'
args.device = device
args.num_classes = len(busi_classes)

# Training knobs.
args.epochs = 30
args.batch_size = 8
args.lr = 1e-4
args.accumulation_steps = 4
args.patience = 15

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1
args.encoder = 'vision'

save_dir = lora_base_save_dir / args.model_name

openai_lora_results_df, openai_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

openai_lora_summary_df


LoRA few-shot: model=openai_clip | k=[1] | seeds=[1] | rank=16 | alpha=32 | dropout=0.1 | layers=None
injected lora adapters to 12 layers (clip vision encoder)
trainable params: 1,179,648 / 150,800,385

LoRA model check: openai_clip
  trainable vision tensors: 96
  vision trainable params: 1,179,648 / 150,800,385
  classifier trainable params: 1,539
  total trainable params: 1,181,187



openai_clip k=1 seed=1:  67%|██████▋   | 20/30 [00:36<00:18,  1.82s/it, best=0.4292, loss=0.7739, lr=2.1e-05, val_f1=0.3661]


done model=openai_clip k=1 seed=1 best_epoch=6 best_val_f1=0.4292 test_acc=0.2211 test_f1=0.2232 test_auc=0.4443

LoRA summary:
 k  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1            0.221053                NaN             0.22321                NaN       0.444262           NaN


,k,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,0.221053,NaN,0.22321,NaN,0.444262,NaN


In [14]:
# Run LoRA training for BiomedCLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False

args.model_name = 'biomedclip_vit_b16'
args.device = device
args.num_classes = len(busi_classes)

# Training knobs.
args.epochs = 30
args.batch_size = 8
args.lr = 1e-4
args.accumulation_steps = 4
args.patience = 15

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1
args.encoder = 'vision'

save_dir = lora_base_save_dir / args.model_name

biomedclip_lora_results_df, biomedclip_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

biomedclip_lora_summary_df


LoRA few-shot: model=biomedclip | k=[1] | seeds=[1] | rank=16 | alpha=32 | dropout=0.1 | layers=None
trainable params: 884,736 || all params: 196,787,457 || trainable%: 0.4496

LoRA model check: biomedclip
  trainable vision tensors: 48
  vision trainable params: 884,736 / 196,787,457
  classifier trainable params: 1,539
  total trainable params: 886,275



biomedclip k=1 seed=1: 100%|██████████| 30/30 [00:49<00:00,  1.64s/it, best=0.2999, loss=1.0524, lr=1.0e-08, val_f1=0.2999]


done model=biomedclip k=1 seed=1 best_epoch=27 best_val_f1=0.2999 test_acc=0.4947 test_f1=0.3270 test_auc=0.4145

LoRA summary:
 k  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1            0.494737                NaN            0.327013                NaN       0.414471           NaN


,k,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,0.494737,NaN,0.327013,NaN,0.414471,NaN


In [15]:
# Run LoRA training for UniMed-CLIP.
args = get_args([])
args.console_log = False
args.show_progress = True
args.log_epochs = False

args.model_name = 'unimedclip'
args.device = device
args.project_root = str(project_root)
args.num_classes = len(busi_classes)

# Training knobs.
args.epochs = 30
args.batch_size = 8
args.lr = 1e-4
args.accumulation_steps = 4
args.patience = 15

# LoRA knobs.
args.lora_layers = None
args.lora_rank = 16
args.lora_alpha = 32
args.lora_dropout = 0.1
args.encoder = 'vision'

save_dir = lora_base_save_dir / args.model_name

unimedclip_lora_results_df, unimedclip_lora_summary_df = run_kshot_experiments(
    args=args,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    class_names=lora_class_names,
    ks=shots_per_class,
    seeds=seeds,
    save_dir=save_dir,
    support_indices=shared_support_indices,
)

unimedclip_lora_summary_df


LoRA few-shot: model=unimedclip | k=[1] | seeds=[1] | rank=16 | alpha=32 | dropout=0.1 | layers=None


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 20007.70it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be

injected lora adapters to 12 layers (clip vision encoder)
trainable params: 1,179,648 / 197,083,137

LoRA model check: unimedclip
  trainable vision tensors: 96
  vision trainable params: 1,179,648 / 197,083,137
  classifier trainable params: 1,539
  total trainable params: 1,181,187



unimedclip k=1 seed=1:  53%|█████▎    | 16/30 [00:27<00:24,  1.74s/it, best=0.4091, loss=1.0041, lr=4.0e-05, val_f1=0.3235]


done model=unimedclip k=1 seed=1 best_epoch=2 best_val_f1=0.4091 test_acc=0.3895 test_f1=0.2523 test_auc=0.4571

LoRA summary:
 k  test_accuracy_mean  test_accuracy_std  test_macro_f1_mean  test_macro_f1_std  test_auc_mean  test_auc_std
 1            0.389474                NaN            0.252326                NaN       0.457097           NaN


,k,test_accuracy_mean,test_accuracy_std,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std
0,1,0.389474,NaN,0.252326,NaN,0.457097,NaN
